# Stage 3 -- Model-Ready: Monthly Full Moments

## Input
`Data/Data_Collection/Final/Stage_2/agg_market_monthly_full_moments.parquet` -- aggregated monthly table with five cap-weighted cross-sectional moments per stock factor plus macro monthly factors, keyed on `date`

## Purpose
Applies expanding-window z-standardisation to the aggregated monthly full moments table to produce the final model-ready monthly dataset. The approach is analogous to the daily full moments notebook but uses a 12-month minimum window instead of 252 days, and includes a pre-screening step to identify and drop near-constant columns before z-scoring rather than handling them post-hoc.

---

## Pipeline

### Step 1: Load
The Stage 2 aggregated monthly full moments table is loaded and sorted by date. Column counts are reported separately for stock moment columns (identified by `_cwmean`, `_cwstd`, `_cwskew`, `_cwkurt`, `_spread` suffixes) and macro/other columns.

### Step 2: Identify Columns to Z-Score vs Skip
Unlike the daily tables, the monthly tables contain no binary or calendar features, so only `date` and `target_monthly_return` are skipped. All remaining columns are z-scored.

**Pre-screening for near-constant columns:** before z-scoring, every candidate column is simulated through the expanding std + shift(1) computation. Columns where more than 5 months post-warmup would have σ ≈ 0 or NaN are dropped. This is more proactive than the daily approach (which dropped only `dlyreti_spread` explicitly) because monthly data contains a higher proportion of rare-event indicators (corporate event flags, binary OAP signals) that may be constant for extended stretches. Each dropped column is reported with its count of bad post-warmup months. Surviving columns are reported split into stock moment columns and macro/other columns.

### Step 3: Expanding-Window Z-Standardisation
Applied to all `zscore_cols` using the formula:

`z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}`

- **`shift(1)` applied to both expanding mean and std** -- the current month is excluded from its own standardisation, preventing look-ahead
- **Minimum 12 months (~1 year)** before the first valid z-score is produced
- Computed in one vectorised pass using pandas `expanding().mean()` and `expanding().std()` followed by `shift(1)`
- Expanding arrays deleted immediately after use to free memory
- Any resulting ±inf values (from σ = 0 periods) are replaced with NaN

### Step 4: Drop Warmup Rows
The first `MIN_WINDOW + 1` rows (13 rows: 12-month expanding warmup + 1 for the shift) are dropped. If any z-scored columns still contain NaN after this trim, an extended diagnostic is run: each offending column is reported with its last NaN row index and date, and rows are trimmed until all NaN are eliminated.

### Step 5: Validate
- **NaN check:** zero NaN expected in all feature columns after warmup trim; any remaining are listed
- **Infinite value check:** confirms no ±inf remain
- **Zero-variance check:** identifies any columns that are all-NaN or constant after z-scoring
- **Target integrity:** mean (~0.008--0.010), std (~0.04--0.05), min, max, annualised Sharpe, NaN count -- confirms target was not z-scored
- **Z-score distribution check:** for one sample base factor, all five moment columns are shown with their post-z-score mean, std, min, max (expect mean ≈ 0, std ≈ 1); three macro features also shown
- **No duplicate dates**

### Step 6: Save
Sorted by date and saved to parquet.

---

## Key Design Decisions
- **`MIN_WINDOW = 12` months** (vs 252 days for daily tables), reflecting the monthly cadence. One year of history is required before the first z-score is produced.
- **Pre-screening step unique to this notebook:** the monthly cross-section contains more near-constant columns than the daily cross-section (corporate event indicators, rare OAP binary signals) because these events are sparse across ~222 months. Pre-screening and dropping these before z-scoring is cleaner than discovering them post-hoc via ±inf or extended NaN.
- **No binary/calendar skip list needed:** the monthly aggregated table contains no binary or calendar features -- those are Panel C (macro daily) constructs that do not appear in Panel B (stock monthly) or Panel D (macro monthly).
- **Extended NaN diagnostic in warmup trim:** because the pre-screening uses a threshold of >5 bad months (rather than dropping everything with any bad months), a small number of columns may still produce a few early NaN after the standard warmup drop. The diagnostic identifies exactly which columns are responsible and trims rows accordingly.
- **`rare_drop` referenced in final summary print** -- this variable is set by the pre-screening step and contains the list of dropped near-constant columns.

## Output
`Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_monthly_full_moments.parquet` -- keyed on `date` (calendar month-end), all continuous features expanding-window z-standardised using only past data, `target_monthly_return` in raw returns

In [3]:
# %% [markdown]
# # Stage 3 — Model-Ready: Monthly Full Moments
#
# Applies expanding-window z-standardisation to the aggregated monthly full
# moments table (cwmean, cwstd, cwskew, cwkurt, spread per stock factor
# + macro factors), producing the model-ready monthly dataset.
#
# Z-scoring approach:
#   z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}
#   - Uses ONLY data up to t-1 (shift(1) ensures no look-ahead)
#   - Minimum 12 months (~1 year) before first valid z-score
#   - Target variable is NOT z-scored (stays in raw returns)
#
# Input:  Stage_2/agg_market_monthly_full_moments.parquet
# Output: Stage_3_Model_Ready/model_market_monthly_full_moments.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import time

IN_PATH = Path('../../../../Data/Data_Collection/Final/Stage_2/agg_market_monthly_full_moments.parquet')
OUT_DIR = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD")
print("=" * 90)

df = pd.read_parquet(IN_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"\n  Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# Column type breakdown
moment_suffixes = ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']
moment_cols = [c for c in df.columns if any(c.endswith(s) for s in moment_suffixes)]
non_moment_cols = [c for c in df.columns if c not in moment_cols and c not in ['date', 'target_monthly_return']]
print(f"  Stock moment columns: {len(moment_cols)}")
print(f"  Macro/other columns: {len(non_moment_cols)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP")
print("=" * 90)

# Monthly tables have no binary/calendar features — skip only meta
meta_cols = ['date', 'target_monthly_return']
all_skip = {c for c in meta_cols if c in df.columns}
zscore_cols = [c for c in df.columns if c not in all_skip]

print(f"\n  Total columns: {df.shape[1]}")
print(f"  Columns to z-score: {len(zscore_cols)}")
print(f"  Columns to skip: {len(all_skip)} ({sorted(all_skip)})")
# Pre-screen: identify columns that will produce extended NaN z-scores
# by simulating the expanding std and checking where it stays at zero
print(f"\n  Pre-screening for near-constant columns...")
pre_screen_drop = []

MIN_WINDOW = 12

for col in zscore_cols:
    vals = df[col].values.astype(float)
    exp_std = pd.Series(vals).expanding(min_periods=MIN_WINDOW).std().shift(1)
    # After warmup (row MIN_WINDOW), how many rows still have σ ≈ 0 or NaN?
    post_warmup_std = exp_std.iloc[MIN_WINDOW + 1:]
    bad_rows = ((post_warmup_std == 0) | post_warmup_std.isna()).sum()
    if bad_rows > 5:  # more than 5 months of undefined z-scores
        pre_screen_drop.append((col, int(bad_rows)))

if pre_screen_drop:
    drop_cols = [c for c, _ in pre_screen_drop]
    df = df.drop(columns=drop_cols)
    zscore_cols = [c for c in zscore_cols if c not in drop_cols]
    print(f"  Dropped {len(drop_cols)} columns with extended σ ≈ 0:")
    for c, n in sorted(pre_screen_drop, key=lambda x: -x[1]):
        print(f"    {c:<50s} {n} bad months post-warmup")
else:
    print(f"  ✓ All columns have sufficient variation for z-scoring")
# Breakdown
zscore_moment = [c for c in zscore_cols if any(c.endswith(s) for s in moment_suffixes)]
zscore_macro = [c for c in zscore_cols if c not in zscore_moment]
print(f"\n  Z-scored breakdown:")
print(f"    Stock moment columns: {len(zscore_moment)}")
print(f"    Macro/other columns:  {len(zscore_macro)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: EXPANDING-WINDOW Z-STANDARDISATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: EXPANDING-WINDOW Z-STANDARDISATION")
print("=" * 90)

MIN_WINDOW = 12

t0 = time.time()

print(f"\n  Z-scoring {len(zscore_cols)} columns with expanding window (min {MIN_WINDOW} months)...")
print(f"  Formula: z_t = (x_t - μ_{{1:t-1}}) / σ_{{1:t-1}}")
print(f"  shift(1) ensures NO look-ahead — current month excluded from mean/std\n")

expanding_mean = df[zscore_cols].expanding(min_periods=MIN_WINDOW).mean().shift(1)
expanding_std = df[zscore_cols].expanding(min_periods=MIN_WINDOW).std().shift(1)

df[zscore_cols] = (df[zscore_cols] - expanding_mean) / expanding_std

del expanding_mean, expanding_std
import gc; gc.collect()

elapsed = time.time() - t0
print(f"  Z-scoring completed in {elapsed:.1f}s")

# Replace inf from σ = 0
inf_before = np.isinf(df[zscore_cols]).sum().sum()
if inf_before > 0:
    df[zscore_cols] = df[zscore_cols].replace([np.inf, -np.inf], np.nan)
    print(f"  ⚠ Replaced {inf_before} infinite values with NaN (from σ = 0 periods)")
else:
    print(f"  ✓ No infinite values produced")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: DROP WARMUP ROWS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: DROP WARMUP ROWS")
print("=" * 90)

warmup_needed = MIN_WINDOW + 1

pre_drop = len(df)
warmup_date = df.iloc[warmup_needed - 1]['date']
df = df.iloc[warmup_needed:].reset_index(drop=True)

print(f"\n  Dropped first {warmup_needed} rows (warmup period)")
print(f"  Warmup end date: {warmup_date.date()}")

# Check for additional NaN from near-constant factors
nan_rows = df[df[zscore_cols].isna().any(axis=1)]
if len(nan_rows) > 0:
    last_nan_row = max(nan_rows.index)
    pre = len(df)

    # Diagnose which columns are causing extended NaN
    print(f"\n  Diagnosing extended NaN (beyond warmup):")
    for c in zscore_cols:
        if c in df.columns:
            last_nan = df[df[c].isna()].index
            if len(last_nan) > 0:
                max_row = int(last_nan.max())
                nan_date = df.iloc[max_row]['date'].date()
                total_nan = df[c].isna().sum()
                print(f"    {c:<50s} last NaN row {max_row:>4d} ({nan_date})  total: {total_nan}")

    df = df.iloc[last_nan_row + 1:].reset_index(drop=True)
    print(f"\n  Trimmed {pre - len(df)} additional rows to remove early z-score NaN")
else:
    print(f"  ✓ No additional NaN rows to trim")

print(f"\n  Rows: {pre_drop} → {len(df)}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: VALIDATE")
print("=" * 90)

# 5a. NaN check
feature_cols = [c for c in df.columns if c not in ['date', 'target_monthly_return']]
feature_nan = df[feature_cols].isna().sum()
feature_nan_total = feature_nan.sum()

if feature_nan_total > 0:
    nan_cols = feature_nan[feature_nan > 0].sort_values(ascending=False)
    print(f"\n  ⚠ Feature NaN: {feature_nan_total}")
    print(f"  Columns with NaN ({len(nan_cols)}):")
    for c in nan_cols.head(20).index:
        print(f"    {c}: {int(nan_cols[c])}")
else:
    print(f"\n  ✓ Zero NaN in features")

# 5b. Infinite values
inf_count = 0
inf_cols = []
for c in zscore_cols:
    if c in df.columns:
        n_inf = np.isinf(df[c]).sum()
        if n_inf > 0:
            inf_count += n_inf
            inf_cols.append((c, n_inf))

if inf_count > 0:
    print(f"\n  ⚠ Infinite values: {inf_count}")
    for c, n in inf_cols[:15]:
        print(f"    {c}: {n}")
else:
    print(f"  ✓ Zero infinite values")

# 5c. Zero-variance
zero_var_cols = []
for c in zscore_cols:
    if c in df.columns:
        if df[c].isna().all():
            zero_var_cols.append(c)
        elif pd.notna(df[c].std()) and float(df[c].std()) == 0:
            zero_var_cols.append(c)

if zero_var_cols:
    print(f"\n  ⚠ Zero-variance after z-score ({len(zero_var_cols)}):")
    for c in zero_var_cols:
        print(f"    {c}")
else:
    print(f"  ✓ No zero-variance columns")

# 5d. Target untouched
print(f"\n  Target statistics (should be raw returns, NOT z-scored):")
print(f"    Mean:   {df['target_monthly_return'].mean():.6f} (expect ~0.008-0.010)")
print(f"    Std:    {df['target_monthly_return'].std():.6f} (expect ~0.04-0.05)")
print(f"    Min:    {df['target_monthly_return'].min():.6f}")
print(f"    Max:    {df['target_monthly_return'].max():.6f}")
print(f"    Sharpe: {df['target_monthly_return'].mean() / df['target_monthly_return'].std() * np.sqrt(12):.2f} (annualised)")
print(f"    NaN:    {df['target_monthly_return'].isna().sum()}")

# 5e. Z-score distribution (one factor across all moment types)
print(f"\n  Z-score distribution check (one factor across all moments):")
print(f"  {'Column':<50s} {'Mean':>8s} {'Std':>8s} {'Min':>8s} {'Max':>8s}")
print("  " + "-" * 80)

sample_base = None
for c in df.columns:
    if c.endswith('_cwmean') and not c.startswith('stock_'):
        sample_base = c.replace('_cwmean', '')
        break

if sample_base:
    for suffix in ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']:
        col = f'{sample_base}{suffix}'
        if col in df.columns:
            vals = df[col].dropna()
            if len(vals) > 0:
                print(f"  {col:<50s} {vals.mean():>8.3f} {vals.std():>8.3f} "
                      f"{vals.min():>8.2f} {vals.max():>8.2f}")

# Also show a few macro features
macro_samples = [c for c in zscore_macro if c in df.columns][:3]
for c in macro_samples:
    vals = df[c].dropna()
    if len(vals) > 0:
        print(f"  {c:<50s} {vals.mean():>8.3f} {vals.std():>8.3f} "
              f"{vals.min():>8.2f} {vals.max():>8.2f}")

# 5f. No duplicate dates
n_dupes = df['date'].duplicated().sum()
assert n_dupes == 0, "FATAL: Duplicate dates!"
print(f"\n  ✓ No duplicate dates")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: SAVE")
print("=" * 90)

df = df.sort_values('date').reset_index(drop=True)

out_path = OUT_DIR / 'model_market_monthly_full_moments.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')

file_size = out_path.stat().st_size
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]} rows × {df.shape[1]} columns")
print(f"    Size: {file_size / 1e6:.1f} MB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("MODEL-READY MONTHLY FULL MOMENTS COMPLETE")
print("=" * 90)

n_zscored = len([c for c in zscore_cols if c in df.columns])

print(f"""
  Input:  agg_market_monthly_full_moments.parquet (Stage 2)
  Output: model_market_monthly_full_moments.parquet (Stage 3)

  Z-standardisation:
    Method:     Expanding window, shift(1), min {MIN_WINDOW} months
    Z-scored:   {n_zscored} features ({len(zscore_moment)} stock moments + {len(zscore_macro)} macro)
    Skipped:    {len(all_skip)} (date + target)
    Dropped:    {len(rare_drop)} near-constant rare-event columns
    Warmup:     {warmup_needed}+ rows dropped

  Result:
    Rows:       {df.shape[0]} months
    Columns:    {df.shape[1]}
    Dates:      {df['date'].min().date()} → {df['date'].max().date()}
    NaN:        {feature_nan_total} features + {df['target_monthly_return'].isna().sum()} target

  Saved: {out_path}
""")

STEP 1: LOAD

  Loaded: 218 rows × 1083 columns
  Date range: 2006-10-31 → 2024-11-30
  Stock moment columns: 948
  Macro/other columns: 133

STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP

  Total columns: 1083
  Columns to z-score: 1081
  Columns to skip: 2 (['date', 'target_monthly_return'])

  Pre-screening for near-constant columns...
  Dropped 14 columns with extended σ ≈ 0:
    DivInit_spread                                     205 bad months post-warmup
    DivOmit_spread                                     205 bad months post-warmup
    DivSeason_spread                                   205 bad months post-warmup
    ExchSwitch_spread                                  205 bad months post-warmup
    IndIPO_spread                                      205 bad months post-warmup
    Spinoff_spread                                     205 bad months post-warmup
    rec_median_spread                                  205 bad months post-warmup
    analyst_alignment_spread                 